# IT25104113 — Support Vector Machine (SVM)

**AIML Group Assignment** | **Dataset:** UCI Default of Credit Card Clients

## Setup — shared train/test split

This notebook is a standalone copy of this member's section from `group_pipeline.ipynb`. The model code below is unchanged.

This cell loads the same 80/20 split that `group_pipeline.ipynb` saves to `results/outputs/train.csv` and `results/outputs/test.csv`, and defines the same `record_model()` function, so the scores match the group notebook. Run `group_pipeline.ipynb` first if these files are missing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

TARGET = 'default payment next month'

# Same rows as the shared split in group_pipeline.ipynb.
# float_precision='round_trip' reads every value back exactly as it was saved.
train_df = pd.read_csv('../results/outputs/train.csv', float_precision='round_trip')
test_df = pd.read_csv('../results/outputs/test.csv', float_precision='round_trip')
X_train, y_train = train_df.drop(columns=TARGET), train_df[TARGET]
X_test, y_test = test_df.drop(columns=TARGET), test_df[TARGET]
print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows')

model_results = {}

def record_model(student_id, model_name, fitted_model):
    """Evaluate a fitted binary classifier on the shared test set."""
    predictions = fitted_model.predict(X_test)
    result = {
        'IT Number': student_id,
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'F1': f1_score(y_test, predictions, zero_division=0),
        'ROC-AUC': 'N/A',
    }
    classes = list(getattr(fitted_model, 'classes_', []))
    if 1 in classes and hasattr(fitted_model, 'predict_proba'):
        scores = fitted_model.predict_proba(X_test)[:, classes.index(1)]
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    elif 1 in classes and hasattr(fitted_model, 'decision_function'):
        scores = fitted_model.decision_function(X_test)
        if len(classes) == 2 and classes.index(1) == 0:
            scores = -scores
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    model_results[student_id] = result
    return pd.Series(result)


### IT25104113 — Support Vector Machine (SVM)

**Model suitability**
- The task is binary classification: will the client default next month (1) or not (0)?
- SVM finds the boundary that separates the two classes with the widest possible margin, which makes it robust to noisy points.
- With different kernels it can learn a straight-line boundary (`linear`) or a curved one (`rbf`). Repayment behaviour (e.g. `PAY_0`, `delay_count`) relates to default in a non-linear way, so testing both kernels is useful.

**Implementation**
- Library: scikit-learn (`SVC`, `StandardScaler`, `make_pipeline`, `GridSearchCV`, `StratifiedKFold`).
- SVM depends on distances between points, so every feature must be on the same scale. `StandardScaler` is placed **inside a pipeline**, so it is fitted only on the training folds and never sees the test data.
- `gamma='scale'` (scikit-learn's default) sets the width of the RBF kernel automatically.

**Parameter tuning**
- Baseline: RBF SVM with default settings (`C=1`, no class weighting), for comparison.
- Method: `GridSearchCV`, trying every combination below (3 × 2 × 2 = **12 models**):
  - `C` (how strictly the margin punishes mistakes): 0.1, 1, 10
  - `kernel`: `linear` or `rbf`
  - `class_weight`: `None` or `balanced` (only ~22% of clients default, so `balanced` makes the model pay more attention to defaulters)
- The grid search takes around 12 minutes because SVM training is slow on 24,000 rows.

**Validation method**
- 3-fold **stratified** cross-validation on the training set only. Each fold keeps the same 22% default ratio.
- The best model is chosen by **cross-validation F1**, not by test score. It is then checked once on the held-out 20% test set.

**Evaluation metrics**
- **F1** (used to choose the best model): balances precision and recall. Accuracy is misleading here because predicting "no default" for everyone would already be ~78% accurate.
- **Recall**: how many real defaulters we catch. Missing a defaulter is costly for a bank.
- **Precision**: when we predict default, how often we are right.
- **ROC-AUC**: how well the model ranks risky clients above safe ones, regardless of threshold.

In [ ]:
# IT25104113 — Support Vector Machine (SVM) with grid search
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay

# 1. Baseline: RBF SVM with default settings (no tuning), kept for comparison.
#    StandardScaler sits inside the pipeline, so it is fitted on training data only.
baseline_svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1, gamma='scale'))
baseline_svm.fit(X_train, y_train)

# 2. Settings to try: 3 C values x 2 kernels x 2 class weights = 12 models
param_grid = {
    'svc__C': [0.1, 1, 10],
    'svc__kernel': ['linear', 'rbf'],
    'svc__class_weight': [None, 'balanced'],
}

# 3. 3-fold stratified cross-validation on the training data only (takes ~12 minutes)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
svm_search = GridSearchCV(
    make_pipeline(StandardScaler(), SVC(gamma='scale')),
    param_grid=param_grid,
    scoring='f1',
    cv=cv,
    n_jobs=-1
)
svm_search.fit(X_train, y_train)
best_svm = svm_search.best_estimator_
print(f'Best settings: {svm_search.best_params_}')
print(f'Best cross-validation F1: {svm_search.best_score_:.3f}')

# 4. Compare all 12 models tried
results = pd.DataFrame(svm_search.cv_results_)
results = results[['param_svc__C', 'param_svc__kernel', 'param_svc__class_weight', 'mean_test_score']]
results.columns = ['C', 'Kernel', 'Class Weight', 'CV F1']
results['Class Weight'] = results['Class Weight'].fillna('None')
display(results.sort_values('CV F1', ascending=False).reset_index(drop=True))

# 5. Final check on the test set: baseline vs tuned
y_pred_base = baseline_svm.predict(X_test)
y_pred_svm = best_svm.predict(X_test)
test_comparison = pd.DataFrame({
    'Model': ['Baseline SVM (RBF, C=1)', 'Tuned SVM (best)'],
    'Accuracy': [accuracy_score(y_test, y_pred_base), accuracy_score(y_test, y_pred_svm)],
    'Precision': [precision_score(y_test, y_pred_base), precision_score(y_test, y_pred_svm)],
    'Recall': [recall_score(y_test, y_pred_base), recall_score(y_test, y_pred_svm)],
    'F1': [f1_score(y_test, y_pred_base), f1_score(y_test, y_pred_svm)],
    'ROC-AUC': [roc_auc_score(y_test, baseline_svm.decision_function(X_test)),
                roc_auc_score(y_test, best_svm.decision_function(X_test))],
})
display(test_comparison.round(4))
print(classification_report(y_test, y_pred_svm, target_names=['No Default', 'Default']))

# 6. Confusion matrix and ROC curve
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_svm,
                                        display_labels=['No Default', 'Default'], ax=axes[0])
axes[0].set_title('IT25104113 Tuned SVM: Confusion Matrix')
RocCurveDisplay.from_estimator(baseline_svm, X_test, y_test, name='Baseline SVM', ax=axes[1])
RocCurveDisplay.from_estimator(best_svm, X_test, y_test, name='Tuned SVM', ax=axes[1])
axes[1].plot([0, 1], [0, 1], '--', color='gray', label='Random guess')
axes[1].legend()
axes[1].set_title('IT25104113 SVM: ROC Curve')
plt.tight_layout()
plt.show()

# 7. Support vectors: training points that define the boundary
n_support = best_svm.named_steps['svc'].n_support_
print(f'Support vectors: {n_support[0]:,} No Default + {n_support[1]:,} Default '
      f'= {n_support.sum():,} of {len(X_train):,} training rows')

record_model('IT25104113', 'Support Vector Machine (SVM)', best_svm)

**IT25104113 — Results and conclusion**

- **Comparison of the 12 models:** every model with `class_weight='balanced'` (CV F1 0.51–0.54) beat every model without it (CV F1 0.39–0.47). The `rbf` kernel beat `linear` in both groups, so the relationship between the features and default is not a straight line. For the linear kernel, changing `C` made no difference.
- **Best model:** `kernel='rbf'`, `C=1`, `class_weight='balanced'` (CV F1 ≈ 0.541).
- **Baseline vs tuned (test set):**

| Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Baseline SVM (RBF, C=1) | 0.817 | 0.671 | 0.342 | 0.453 | 0.712 |
| Tuned SVM (best) | 0.771 | 0.485 | **0.588** | **0.532** | **0.755** |

- **Conclusion:** the baseline looked good on accuracy (82%) but caught only 34% of defaulters. Class balancing raised recall to about 59% and F1 from 0.45 to 0.53, with a small drop in accuracy and precision. That trade-off is worth it because missing a defaulter costs the bank more than double-checking a safe client.
- **Support vectors:** about 14,700 of the 24,000 training rows are support vectors. Many points sit near the boundary, which shows the two classes overlap a lot in this data.

**Limitations and possible improvements**
- SVM is slow on large data. The 12-model grid takes ~12 minutes, so a wider search (e.g. tuning `gamma`) would be costly.
- SVM does not give feature importances for the `rbf` kernel, so it is harder to explain than Logistic Regression or a Decision Tree.
- The decision threshold is fixed at 0. Moving it could trade precision for recall more precisely.
- Preprocessing (Stages 1–6) was fitted before the train/test split, so test scores may be slightly optimistic.